# SNNAI Phase0.5 — Liquid State Machine (LSM) Reproduction

This notebook reproduces the core idea of Liquid State Machines (Maass et al., 2002):
a randomly connected recurrent network of spiking neurons (the **liquid**) is used as a reservoir,
and a simple linear readout is trained on the reservoir states.

**Goal**: classify two synthetic time-series patterns using a snnTorch LIF reservoir.

In [ ]:
# Install dependencies (compatible with Kaggle default torch/numpy)
!pip install -q snntorch==0.9.4 scikit-learn==1.6.0

In [ ]:
import numpy as np
import torch
import snntorch as snn
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

## 1. Generate synthetic time-series data

We create two classes of 1-D signals:
- Class 0: low-frequency sine wave
- Class 1: high-frequency sine wave

Each sample has `T=200` time steps.

In [ ]:
def generate_signal(freq, T=200, noise=0.05):
    t = np.linspace(0, 2 * np.pi, T)
    return np.sin(freq * t) + noise * np.random.randn(T)

n_samples = 400
T = 200
X = []
y = []
for _ in range(n_samples // 2):
    X.append(generate_signal(freq=1.0, T=T))
    y.append(0)
for _ in range(n_samples // 2):
    X.append(generate_signal(freq=3.0, T=T))
    y.append(1)

X = np.array(X)
y = np.array(y)
print('Data shape:', X.shape, y.shape)

## 2. Rate-code the signals into spike trains

Each time-step value is scaled to a firing probability.

In [ ]:
def rate_encode(X, max_rate=0.8, n_neurons=20):
    n_samples, T = X.shape
    gains = np.random.uniform(0.5, 1.5, size=(1, 1, n_neurons))
    X_norm = (X - X.min()) / (X.max() - X.min() + 1e-8)
    rates = X_norm[:, :, None] * gains * max_rate
    spikes = (np.random.rand(n_samples, T, n_neurons) < rates).astype(float)
    return torch.tensor(spikes, dtype=torch.float32)

n_input = 20
spike_data = rate_encode(X, n_neurons=n_input)
print('Spike tensor shape:', spike_data.shape)

## 3. Build the liquid reservoir

- Input → Reservoir (random feedforward + recurrent weights)
- Reservoir neurons are snnTorch LIF neurons
- We do **not** train reservoir weights; they are fixed randomly.

In [ ]:
n_reservoir = 100
beta = 0.9
threshold = 1.0

w_in = torch.randn(n_input, n_reservoir) * 0.5

# Sparse random recurrent weights scaled to spectral radius < 1
w_rec = torch.randn(n_reservoir, n_reservoir)
mask = torch.rand(n_reservoir, n_reservoir) < 0.1
w_rec = w_rec * mask.float()
w_rec.fill_diagonal_(0)

# Normalize by spectral radius
with torch.no_grad():
    eigvals = torch.linalg.eigvals(w_rec)
    spectral_radius = torch.max(torch.abs(eigvals)).item()
    if spectral_radius > 0:
        w_rec = w_rec * (0.9 / spectral_radius)

lif = snn.Leaky(beta=beta, threshold=threshold, learn_threshold=False)
print('Reservoir size:', n_reservoir)

## 4. Run the reservoir and collect states

In [ ]:
def run_reservoir(spike_sample):
    mem = lif.init_leaky()
    if mem.dim() == 0:
        mem = torch.zeros(n_reservoir)
    spikes = []
    for t in range(spike_sample.shape[0]):
        input_current = spike_sample[t] @ w_in
        rec_current = spikes[-1] @ w_rec if spikes else torch.zeros(n_reservoir)
        current = input_current + rec_current
        spk, mem = lif(current, mem)
        spikes.append(spk.detach())
    return torch.stack(spikes)

# Collect average firing rates from the reservoir
all_rates = []
for i in range(len(spike_data)):
    spikes = run_reservoir(spike_data[i])
    all_rates.append(spikes[-50:].mean(dim=0).numpy())  # average over last 50 steps

features = np.array(all_rates)
print('Feature matrix shape:', features.shape)
print('Feature min/max:', features.min(), features.max())

## 5. Train a linear readout and evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, y, test_size=0.3, random_state=42, stratify=y
)

clf = RidgeClassifier(alpha=1.0)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.4f}')

## 6. Visualize reservoir separability

In [ ]:
# Text-based reservoir separability analysis
from sklearn.decomposition import PCA

pca = PCA(n_components=5)
features_pca = pca.fit_transform(features)

print('PCA explained variance ratio:', pca.explained_variance_ratio_.round(4))
print('Cumulative explained variance:', pca.explained_variance_ratio_.cumsum().round(4))
print()
# Class-wise mean firing rates
for label in [0, 1]:
    idx = y == label
    mean_rate = features[idx].mean()
    print(f'Class {label}: mean reservoir firing rate = {mean_rate.mean():.4f}')

# Distance between class centroids in PCA space
c0 = features_pca[y == 0].mean(axis=0)
c1 = features_pca[y == 1].mean(axis=0)
dist = np.linalg.norm(c0 - c1)
print(f'Centroid distance in PCA space: {dist:.4f}')